# От CSV к отчёту по заказам
Разобранный пример. Для своего проекта повторите расчёт на project-orders/project-refunds. Отдельно сохраняйте исходники и неизвестные суммы.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Откройте блокнот из папки комплекта"
sys.path.insert(0, str(ROOT))
import pandas as pd
from analysis import basic, independent_basic, normalized, export_basic


In [2]:
orders = pd.read_csv(ROOT / "data/orders.csv", dtype={"order_id":"string", "customer_id":"string"})
print(orders.shape)
display(orders.head())
orders.info()

(6, 6)


,order_id,customer_id,amount_cents,currency,status,created_at
0,1,1,1000.0,RUB,paid,2026-01-01T10:00:00Z
1,2,1,2000.0,RUB,paid,2026-01-02T10:00:00Z
2,3,2,0.0,RUB,paid,2026-01-02T12:00:00Z
3,4,3,500.0,USD,paid,2026-01-03T10:00:00Z
4,5,2,700.0,RUB,cancelled,2026-01-03T11:00:00Z


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      6 non-null      string 
 1   customer_id   6 non-null      string 
 2   amount_cents  5 non-null      float64
 3   currency      6 non-null      object 
 4   status        6 non-null      object 
 5   created_at    6 non-null      object 
dtypes: float64(1), object(3), string(2)
memory usage: 420.0+ bytes


In [3]:
mask = (orders["status"] == "paid") & (orders["currency"] == "RUB")
selected = orders.loc[mask].copy()
print("rows:",len(selected), "unknown:",selected.amount_cents.isna().sum())
display(selected)

rows: 4 unknown: 1


,order_id,customer_id,amount_cents,currency,status,created_at
0,1,1,1000.0,RUB,paid,2026-01-01T10:00:00Z
1,2,1,2000.0,RUB,paid,2026-01-02T10:00:00Z
2,3,2,0.0,RUB,paid,2026-01-02T12:00:00Z
5,6,4,NaN,RUB,paid,2026-01-04T10:00:00Z


## Возвраты до соединения
Два возврата заказа 1 сначала складываются; сумма заказа не повторяется.

In [4]:
refunds = pd.read_csv(ROOT / "data/refunds.csv", dtype={"order_id":"string"})
r = refunds.groupby("order_id",as_index=False).agg(refunded_cents=("amount_cents","sum"))
joined = selected.merge(r,on="order_id",how="left",validate="one_to_one")
joined["refunded_cents"] = joined.refunded_cents.fillna(0)
joined["net_cents"] = joined.amount_cents - joined.refunded_cents
display(joined[["order_id","amount_cents","refunded_cents","net_cents"]])

,order_id,amount_cents,refunded_cents,net_cents
0,1,1000.0,300.0,700.0
1,2,2000.0,300.0,1700.0
2,3,0.0,0.0,0.0
3,6,NaN,0.0,NaN


In [5]:
customers = pd.read_csv(ROOT / "data/customers.csv", dtype={"customer_id":"string"})
joined = joined.merge(customers,on="customer_id",how="left",validate="many_to_one",indicator=True)
print(joined["_merge"].value_counts())

_merge
both          4
left_only     0
right_only    0
Name: count, dtype: int64


## Намеренная ошибка справочника
Пойманная MergeError подтверждает отказ; отключать validate не нужно.

In [6]:
bad_customers = pd.concat([customers,customers.iloc[[0]]],ignore_index=True)
try:
    selected.merge(bad_customers,on="customer_id",how="left",validate="many_to_one")
except pd.errors.MergeError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("Ожидался отказ справочника")

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


In [7]:
result = basic(ROOT)
assert normalized(result) == independent_basic(ROOT)
display(result)
print(export_basic(ROOT))

,order_id,customer_id,channel,currency,amount_cents,refunded_cents,net_cents
0,1,1,search,RUB,1000,300,700
1,2,1,search,RUB,2000,300,1700
2,3,2,email,RUB,0,0,0
3,6,4,<NA>,RUB,<NA>,0,<NA>


{'passed': True, 'rows': 4, 'known_net_cents': 2400, 'unknown_orders': 1, 'sources': {'orders.csv': '5291af02d59d837441a3a1757eaadce1eee6ed2a21f6832f8101705606efc51c', 'refunds.csv': '9371ce4775779cf2f05dcbb088dd6a88550fab04f93f0225c28f81b008340739', 'customers.csv': 'd311e72bd419ed97aceb9725e1ed7c4e6b3f5c39c48b8580ca1da883dad9a3c2'}, 'definition': 'paid RUB, known amounts reported separately from unknown; refunds by order', 'complete': False}


## Ваша очередь
Сделайте my-report.ipynb, прочитав project-файлы. Получите тот же набор выходных полей, сохраните outputs/project-report.csv и запустите локальную check-basic --project. Готовая basic(ROOT,True) — эталон для сверки, не собственное решение проекта.